In [4]:
import os
import cv2
import numpy as np
import pandas as pd
import tensorflow as tf
import seaborn as sns
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

from tensorflow.keras.applications import VGG19
from tensorflow.keras.applications.vgg19 import preprocess_input
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D, Dropout
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.utils import to_categorical

In [ ]:
CSV_PATH = "archive/data.csv"
IMAGE_DIR = "archive/image"

df = pd.read_csv(CSV_PATH)
image_column = df.columns[0]

df["image_path"] = df[image_column].astype(str).apply(
    lambda x: os.path.join(IMAGE_DIR, x + ".jpeg")
)

df = df[df["image_path"].apply(os.path.exists)]
print("Total samples:", len(df))

Total samples: 1512


In [6]:
encoder = LabelEncoder()
df["label_encoded"] = encoder.fit_transform(df["classes"])
print("Classes:", encoder.classes_)

Classes: ['bumper_dent' 'bumper_scratch' 'door_dent' 'door_scratch' 'glass_shatter'
 'head_lamp' 'tail_lamp' 'unknown']


In [7]:
IMG_SIZE = 128  # Reduced from 224 to save memory
images = []
labels = []
for _, row in df.iterrows():
    img = cv2.imread(row["image_path"])
    if img is None:
        continue
    img = cv2.resize(img, (IMG_SIZE, IMG_SIZE))
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    img = img.astype(np.float32) / 255.0  # Normalize to [0,1] as float32
    images.append(img)
    labels.append(row["label_encoded"])

X = np.array(images, dtype=np.float32)  # Ensure float32 dtype
y = np.array(labels)
print("Dataset shape:", X.shape)

Dataset shape: (1512, 128, 128, 3)


In [8]:
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.5, stratify=y_temp, random_state=42
)
print("Train:", X_train.shape)
print("Validation:", X_val.shape)
print("Test:", X_test.shape)

num_classes = len(np.unique(y))
y_train_cat = to_categorical(y_train, num_classes)
y_val_cat = to_categorical(y_val, num_classes)
y_test_cat = to_categorical(y_test, num_classes)

Train: (1209, 128, 128, 3)
Validation: (151, 128, 128, 3)
Test: (152, 128, 128, 3)


In [9]:
datagen = ImageDataGenerator(
    rotation_range=20,
    zoom_range=0.2,
    horizontal_flip=True,
    width_shift_range=0.1,
    height_shift_range=0.1
)
# Note: datagen.fit() is not needed since we're not using featurewise normalization

In [10]:
# ─────────────────────────────────────────────────────────────────
# Custom CNN-12 Architecture
# 4 Convolutional Blocks:
#   Conv Block 1 → 2 conv layers x2  (total 4 conv)
#   Conv Block 2 → 4 conv layers x2  (total 8 conv)
#   = 12 convolutional layers
# Each conv uses ReLU + BatchNorm, each block ends with MaxPooling + Dropout
# Fully Connected: 512 nodes → Dropout → Softmax
# Optimizer: Adam | Loss: Categorical Cross-Entropy
# ─────────────────────────────────────────────────────────────────

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Dense, Flatten, Dropout, BatchNormalization

model = Sequential(name="Custom_CNN_12")

# ── Conv Block 1a  (2 conv, 32 filters) ──
model.add(Conv2D(32, (3, 3), padding='same', activation='relu',
                 input_shape=(128, 128, 3)))
model.add(BatchNormalization())
model.add(Conv2D(32, (3, 3), padding='same', activation='relu'))
model.add(BatchNormalization())
model.add(MaxPooling2D((2, 2), strides=2))
model.add(Dropout(0.25))

# ── Conv Block 1b  (2 conv, 64 filters) ──
model.add(Conv2D(64, (3, 3), padding='same', activation='relu'))
model.add(BatchNormalization())
model.add(Conv2D(64, (3, 3), padding='same', activation='relu'))
model.add(BatchNormalization())
model.add(MaxPooling2D((2, 2), strides=2))
model.add(Dropout(0.25))

# ── Conv Block 2a  (4 conv, 128 filters) ──
model.add(Conv2D(128, (3, 3), padding='same', activation='relu'))
model.add(BatchNormalization())
model.add(Conv2D(128, (3, 3), padding='same', activation='relu'))
model.add(BatchNormalization())
model.add(Conv2D(128, (3, 3), padding='same', activation='relu'))
model.add(BatchNormalization())
model.add(Conv2D(128, (3, 3), padding='same', activation='relu'))
model.add(BatchNormalization())
model.add(MaxPooling2D((2, 2), strides=2))
model.add(Dropout(0.25))

# ── Conv Block 2b  (4 conv, 256 filters) ──
model.add(Conv2D(256, (3, 3), padding='same', activation='relu'))
model.add(BatchNormalization())
model.add(Conv2D(256, (3, 3), padding='same', activation='relu'))
model.add(BatchNormalization())
model.add(Conv2D(256, (3, 3), padding='same', activation='relu'))
model.add(BatchNormalization())
model.add(Conv2D(256, (3, 3), padding='same', activation='relu'))
model.add(BatchNormalization())
model.add(MaxPooling2D((2, 2), strides=2))
model.add(Dropout(0.25))

# ── Classifier Head ──
model.add(Flatten())
model.add(Dense(512, activation='relu'))
model.add(Dropout(0.5))
model.add(Dense(num_classes, activation='softmax'))

c:\Users\Bhanu Praneeth G\AppData\Local\Programs\Python\Python310\lib\site-packages\keras\src\layers\convolutional\base_conv.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [11]:
model.compile(
    optimizer=Adam(learning_rate=0.001),  # Increased from 0.0001 for better learning
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)

model.summary()

early_stop = EarlyStopping(
    monitor="val_loss",
    patience=5,
    restore_best_weights=True
)

# Add learning rate reduction on plateau
from tensorflow.keras.callbacks import ReduceLROnPlateau
reduce_lr = ReduceLROnPlateau(
    monitor="val_loss",
    factor=0.5,
    patience=3,
    min_lr=1e-6,
    verbose=1
)

history = model.fit(
    datagen.flow(X_train, y_train_cat, batch_size=32),
    validation_data=(X_val, y_val_cat),
    epochs=50,  # Increased max epochs
    callbacks=[early_stop, reduce_lr]
)

Model: "Custom_CNN_12"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d_12 (Conv2D)              │ (None, 128, 128, 32)   │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_12          │ (None, 128, 128, 32)   │           128 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_13 (Conv2D)              │ (None, 128, 128, 32)   │         9,248 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_13          │ (None, 128, 128, 32)   │           128 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_4 (MaxPooling2D)  │ (None, 64, 64, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_5 (Dropout)             │ (None, 64, 64, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_14 (Conv2D)              │ (None, 64, 64, 64)     │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_14          │ (None, 64, 64, 64)     │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_15 (Conv2D)              │ (None, 64, 64, 64)     │        36,928 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_15          │ (None, 64, 64, 64)     │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_5 (MaxPooling2D)  │ (None, 32, 32, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_6 (Dropout)             │ (None, 32, 32, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_16 (Conv2D)              │ (None, 32, 32, 128)    │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_16          │ (None, 32, 32, 128)    │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_17 (Conv2D)              │ (None, 32, 32, 128)    │       147,584 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_17          │ (None, 32, 32, 128)    │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_18 (Conv2D)              │ (None, 32, 32, 128)    │       147,584 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_18          │ (None, 32, 32, 128)    │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_19 (Conv2D)              │ (None, 32, 32, 128)    │       147,584 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_19          │ (None, 32, 32, 128)    │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_6 (MaxPooling2D)  │ (None, 16, 16, 128)    │             

 Total params: 11,047,720 (42.14 MB)

 Trainable params: 11,044,264 (42.13 MB)

 Non-trainable params: 3,456 (13.50 KB)

Epoch 1/40


c:\Users\Bhanu Praneeth G\AppData\Local\Programs\Python\Python310\lib\site-packages\keras\src\trainers\data_adapters\py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


38/38 ━━━━━━━━━━━━━━━━━━━━ 66s 2s/step - accuracy: 0.1982 - loss: 4.4959 - val_accuracy: 0.0993 - val_loss: 2.3858
Epoch 2/40
38/38 ━━━━━━━━━━━━━━━━━━━━ 54s 1s/step - accuracy: 0.3474 - loss: 1.9869 - val_accuracy: 0.0993 - val_loss: 2.8333
Epoch 3/40
38/38 ━━━━━━━━━━━━━━━━━━━━ 51s 1s/step - accuracy: 0.3779 - loss: 1.8061 - val_accuracy: 0.0993 - val_loss: 3.7059
Epoch 4/40
25/38 ━━━━━━━━━━━━━━━━━━━━ 16s 1s/step - accuracy: 0.4295 - loss: 1.6871

AbortedError: Graph execution error:

Detected at node StatefulPartitionedCall/gradient_tape/Custom_CNN_12_1/conv2d_22_1/convolution/Conv2DBackpropFilter defined at (most recent call last):
<stack traces unavailable>
Operation received an exception:Status: 1, message: could not create a memory object, in file tensorflow/core/kernels/mkl/mkl_conv_grad_filter_ops.cc:685
	 [[{{node StatefulPartitionedCall/gradient_tape/Custom_CNN_12_1/conv2d_22_1/convolution/Conv2DBackpropFilter}}]] [Op:__inference_multi_step_on_iterator_11441]

In [ ]:
test_loss, test_acc = model.evaluate(X_test, y_test_cat)

print("\nModel Evaluation")
print("----------------------------")
print(f"Test Loss     : {test_loss:.4f}")
print(f"Test Accuracy : {test_acc:.4f}")

pred = model.predict(X_test)
pred_classes = np.argmax(pred, axis=1)
y_test_labels = encoder.inverse_transform(y_test)
pred_labels = encoder.inverse_transform(pred_classes)

print("\nOverall Evaluation Metrics")
print("----------------------------")

accuracy = accuracy_score(y_test_labels, pred_labels)
precision = precision_score(y_test_labels, pred_labels, average='weighted')
recall = recall_score(y_test_labels, pred_labels, average='weighted')
f1 = f1_score(y_test_labels, pred_labels, average='weighted')

print(f"Accuracy  : {accuracy:.4f}")
print(f"Precision : {precision:.4f}")
print(f"Recall    : {recall:.4f}")
print(f"F1 Score  : {f1:.4f}")
print("\nDetailed Classification Report")
print("----------------------------")
print(classification_report(y_test_labels, pred_labels))

cm = confusion_matrix(y_test_labels, pred_labels)

print("\nConfusion Matrix")
print("----------------------------")
print(cm)

plt.figure(figsize=(6,5))
sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=encoder.classes_,
    yticklabels=encoder.classes_
)

plt.xlabel("Predicted Label")
plt.ylabel("True Label")
plt.title("Confusion Matrix")
plt.show()

# Custom CNN-12 Layer Model

This notebook implements a custom convolutional neural network with exactly 12 convolutional layers, designed for car damage classification. The architecture is built from scratch without using pre-trained weights, allowing the model to learn features specifically tailored to this dataset.

## Architecture Details:

- **Input Size**: 128×128×3 (reduced from 224×224 to save memory)
- **4 Convolutional Blocks**:
  - Block 1a: 2 conv layers (32 filters each)
  - Block 1b: 2 conv layers (64 filters each)
  - Block 2a: 4 conv layers (128 filters each)
  - Block 2b: 4 conv layers (256 filters each)
- **Total: 12 Convolutional Layers**

- **Regularization**:
  - Batch Normalization after each conv layer
  - Dropout (0.25) after each max-pooling
  - Dropout (0.5) in the classifier head

- **Classifier**: Dense(512) → Dropout(0.5) → Dense(num_classes, softmax)

- **Training**:
  - Adam optimizer with learning rate 0.0001
  - Early stopping with patience 5
  - Data augmentation for better generalization

This custom architecture aims to achieve high accuracy (85%+) by learning hierarchical features from the car damage images through its deep convolutional structure.